<a href="https://colab.research.google.com/github/adriakms/studyingpython/blob/main/gee_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Importar bibliotecas
import ee
import geemap.core as geemap

In [2]:
# Autenticar conta do Google Earth Engine
ee.Authenticate()

In [3]:
# Inicializar projeto do Google Earth Engine
ee.Initialize(project='doc-adria')

In [18]:
# Criar mapa interativo

Map = geemap.Map(center=[-22.00, -47.89], zoom=10)
Map

Map(center=[-22.0, -47.89], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

In [9]:
# 1. Definir área de estudo
sao_carlos = ee.FeatureCollection('projects/ee-bmm-mapbiomas/assets/ibge/BR_Municipios_2021')\
.filter(ee.Filter.eq('NM_MUN', 'São Carlos'))\
.filter(ee.Filter.eq('SIGLA', 'SP'))

sao_carlos

In [20]:
# Visualizar área de estudo

Map.centerObject(sao_carlos, 10)
Map.addLayer(sao_carlos, {'color': 'orange'}, 'São Carlos')
Map

Map(bottom=147737.0, center=[-21.918758006267115, -47.8671487742629], controls=(ZoomControl(options=['position…

In [23]:
# 2. Visualizar dados de sensor
# Harmonized Sentinel-2 MSI: MultiSpectral Instrument, Level-2A (SR)
s2_sao_carlos = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')\
.filterBounds(sao_carlos)\
.filterDate('2025-01-01', '2025-12-31')\
.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

s2_sao_carlos

s2_median = s2_sao_carlos.median().clip(sao_carlos)

imageVisParam = {
    'bands': ['B4', 'B3', 'B2'],  # RGB
    'min': 111,
    'max': 2556,
    'gamma': 1,
    'opacity': 1
}

Map.addLayer(s2_median, imageVisParam, 'Sentinel-2 de São Carlos em 2025')


In [24]:
# 3. Cor verdadeira e falsa cor

trueColor = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}
falseColor = {'bands': ['B8', 'B4', 'B3'], 'min': 0, 'max': 3000}

Map.addLayer(s2_median, trueColor, 'Cor Verdadeira (RGB)')
Map.addLayer(s2_median, falseColor, 'Falsa Cor - Vegetação')

In [25]:
# 4. Índices espectrais

ndvi = s2_median.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndwi = s2_median.normalizedDifference(['B3', 'B8']).rename('NDWI')
ndbi = s2_median.expression(
    '(SWIR - NIR) / (SWIR + NIR)', {
        'SWIR': s2_median.select('B11'),
        'NIR': s2_median.select('B8')
    }).rename('NDBI')

ndviVis = {'min': -1, 'max': 1, 'palette': ['brown', 'orange', 'green']}
ndwiVis = {'min': -1, 'max': 1, 'palette': ['brown', 'blue']}
ndbiVis = {'min': -1, 'max': 1, 'palette': ['green', 'gray', 'white']}

Map.addLayer(ndvi, ndviVis, 'NDVI - Vegetação')
Map.addLayer(ndwi, ndwiVis, 'NDWI - Água')
Map.addLayer(ndbi, ndbiVis, 'NDBI - Áreas Urbanas')

In [28]:
# 5. Série temporal NDVI

def addNDVI(img):
    return img.normalizedDifference(['B8', 'B4']).rename('NDVI') \
        .copyProperties(img, ['system:time_start'])

ndviCol = s2_sao_carlos.map(addNDVI)

ndviTrim1 = ndviCol.filterDate('2025-01-01', '2025-03-31').mean().clip(sao_carlos)
ndviTrim2 = ndviCol.filterDate('2025-04-01', '2025-06-30').mean().clip(sao_carlos)
ndviTrim3 = ndviCol.filterDate('2025-07-01', '2025-09-30').mean().clip(sao_carlos)
ndviTrim4 = ndviCol.filterDate('2025-10-01', '2025-12-31').mean().clip(sao_carlos)

ndvi_series = ee.Image.cat([
    ndviTrim1.rename('t1'),
    ndviTrim2.rename('t2'),
    ndviTrim3.rename('t3'),
    ndviTrim4.rename('t4')
])

Map.addLayer(ndvi_series, {}, 'Série Trimestral')
Map.addLayer(ndviTrim1, ndviVis, 'NDVI T1 (Jan–Mar)')
Map.addLayer(ndviTrim4, ndviVis, 'NDVI T4 (Out–Dez)')

In [29]:
# 6. Estatísticas

stats = ndvi.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.minMax(), 'combinado-', True),
    geometry=sao_carlos,
    scale=30,
    maxPixels=1e9
)

print('Estatísticas NDVI - São Carlos (2025):', stats.getInfo())

Estatísticas NDVI - São Carlos (2025): {'NDVI_combinado-max': 0.9356607567172364, 'NDVI_combinado-min': -0.6880146386093321, 'NDVI_mean': 0.5275380795297065}


In [31]:
# 7. Exportar resultados

task = ee.batch.Export.image.toDrive(
    image=ndvi,
    description='NDVI_SaoCarlos_2025',
    folder='GEE_Export',
    fileNamePrefix='NDVI_SaoCarlos_2025',
    region=sao_carlos.geometry(),
    scale=10,
    maxPixels=1e13
)

task.start()
print("Exportação configurada: NDVI_SaoCarlos_2025 → Pasta 'GEE_Export' no Google Drive")

Exportação configurada: NDVI_SaoCarlos_2025 → Pasta 'GEE_Export' no Google Drive


In [32]:
# Mostrar mapa novamente

Map

Map(bottom=294887.0, center=[-21.735894877394603, -48.50057786472327], controls=(ZoomControl(options=['positio…